# 🦅 FALCON CFE — Entrenamiento de IA satelital (con Google Drive)

Entrena el detector de infraestructura de CFE con **GPU gratis**, guardando todo en tu **Google Drive** para que **NO pierdas el progreso si Colab se desconecta**.

**Activa la GPU primero:** Menú *Entorno de ejecución → Cambiar tipo* → **GPU (T4)** → Guardar.

Corre las celdas en orden. Si se desconecta a media faena, ve a la sección **REANUDAR** al final.

## 1. Verificar GPU

In [ ]:
!nvidia-smi

Fri Jul  3 10:12:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Conectar Google Drive (aquí se guarda todo)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/falcon_cfe'
os.makedirs(BASE, exist_ok=True)
print('Todo se guardara en:', BASE)

Mounted at /content/drive
Todo se guardara en: /content/drive/MyDrive/falcon_cfe


## 3. Clonar el repositorio e instalar dependencias

In [ ]:
!git clone https://github.com/Prime119/ProyectoMEC.git
%cd ProyectoMEC
!pip install -q pillow httpx ultralytics

Cloning into 'ProyectoMEC'...
remote: Enumerating objects: 199, done.
remote: Counting objects: 100% (199/199), done.
remote: Compressing objects: 100% (137/137), done.
remote: Total 199 (delta 99), reused 152 (delta 54), pack-reused 0 (from 0)
Receiving objects: 100% (199/199), 211.01 KiB | 6.81 MiB/s, done.
Resolving deltas: 100% (99/99), done.
/content/ProyectoMEC
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.3 MB/s eta 0:00:00


## 4. Preparar dataset en Drive (hasta 2000 imágenes auto-etiquetadas)

Se guarda en Drive, así que **solo lo haces una vez**. Si ya lo generaste antes, puedes saltar esta celda.

In [6]:
!python entrenamiento/preparar_dataset.py --zoom 18 --radio 200 --max-imagenes 30000 --salida /content/drive/MyDrive/falcon_cfe/datos

Cargando infraestructura de OpenStreetMap (una vez, puede tardar)...
[OSM] Región: 638 subestaciones, 124 plantas, 1824 líneas
  OSM región: 638 subestaciones, 8692 generadores

== Zona: planta_BC-CC-RPBC ==
  22 activos recolectados
  imagen 1/2000 — 4 etiquetas
  imagen 2/2000 — 2 etiquetas
  imagen 3/2000 — 4 etiquetas
  imagen 4/2000 — 2 etiquetas
  imagen 5/2000 — 3 etiquetas
  imagen 6/2000 — 1 etiquetas
  imagen 7/2000 — 1 etiquetas
  imagen 8/2000 — 2 etiquetas
  imagen 9/2000 — 2 etiquetas
  imagen 10/2000 — 1 etiquetas
  imagen 11/2000 — 2 etiquetas
  imagen 12/2000 — 2 etiquetas
  imagen 13/2000 — 1 etiquetas
  imagen 14/2000 — 1 etiquetas
  imagen 15/2000 — 1 etiquetas
  imagen 16/2000 — 2 etiquetas

== Zona: planta_BC-GEO-CERRO ==
[OSM] https://overpass.kumi.systems/api/interpreter falló (intento 1): The read operation timed out
  79 activos recolectados
  imagen 17/2000 — 8 etiquetas
  imagen 18/2000 — 4 etiquetas
  imagen 19/2000 — 2 etiquetas
  imagen 20/2000 — 4 etique

## 5. Generar configuración (apuntando al dataset en Drive)

In [7]:
!python entrenamiento/generar_config.py --ruta /content/drive/MyDrive/falcon_cfe/datos

✅ dataset.yaml generado con 24 clases (ruta=/content/drive/MyDrive/falcon_cfe/datos) en: /content/ProyectoMEC/entrenamiento/dataset.yaml

Clases:
   0  hidroelectrica        Hidroeléctrica
   1  eolica                Central Eólica
   2  termoelectrica        Termoeléctrica
   3  solar                 Central Solar FV
   4  nucleoelectrica       Nucleoeléctrica
   5  ciclo_combinado       Ciclo Combinado
   6  carbonifera           Carboeléctrica
   7  subestacion           Subestación
   8  torre_grande          Torre de Transmisión Grande (400kV+)
   9  torre_mediana         Torre de Transmisión Mediana (230kV)
  10  torre_chica           Torre de Transmisión Chica (115kV-)
  11  linea_transmision     Línea de Transmisión
  12  transformador         Transformador
  13  poste_grande          Poste de Transmisión Grande
  14  poste_mediano         Poste de Transmisión Mediano
  15  poste_chico           Poste de Transmisión Chico
  16  medidor               Medidor / Punto de Medición


## 6. Entrenar (guarda checkpoints en Drive)

Con GPU T4 y ~2000 imágenes, 100 épocas tardan ~1-2 h. Los pesos se guardan en Drive cada 10 épocas.

In [8]:
!python entrenamiento/entrenar.py --modelo yolov8m.pt --epochs 100 --imgsz 960 --device 0 --project /content/drive/MyDrive/falcon_cfe/runs

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🚀 Entrenando yolov8m.pt sobre dataset.yaml
   epochs=100, imgsz=960, batch=-1, device=0, project=/content/drive/MyDrive/falcon_cfe/runs
New https://pypi.org/project/ultralytics/8.4.87 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.11.0+cpu 
Traceback (most recent call last):
  File "/content/ProyectoMEC/entrenamiento/entrenar.py", line 112, in <module>
    main()
  File "/content/ProyectoMEC/entrenamiento/entrenar.py", line 107, in main
    entrenar(args.modelo, args.epochs, args.imgsz, args.batch, args.nombre,
  File "/content/ProyectoMEC/entrenamiento/entrenar.py", line 69, in entrenar
    resultados = model.train(**kwargs)
     

## 7. El modelo ya está en tu Drive

Al terminar, el modelo queda en `MyDrive/falcon_cfe/runs/cfe_satelital/weights/best.onnx`.
Descárgalo también a tu PC con la siguiente celda:

In [9]:
import glob, shutil
from google.colab import files
onnx = glob.glob('/content/drive/MyDrive/falcon_cfe/runs/**/best.onnx', recursive=True)
if onnx:
    print('Modelo:', onnx[0])
    shutil.copy(onnx[0], 'cfe_satelital.onnx')
    files.download('cfe_satelital.onnx')
else:
    print('Aun no existe best.onnx; revisa que el entrenamiento haya terminado.')

Aun no existe best.onnx; revisa que el entrenamiento haya terminado.


---
## 🔄 REANUDAR (si Colab se desconectó a media faena)

No pierdes nada: el dataset y los checkpoints están en Drive. Solo corre EN ORDEN:
1. La celda **1** (GPU), **2** (Drive) y **3** (clonar + instalar)
2. La celda **5** (generar config apuntando a Drive)
3. Y luego la celda de aquí abajo (reanuda desde el último checkpoint):

In [10]:
!python entrenamiento/entrenar.py --nombre cfe_satelital --project /content/drive/MyDrive/falcon_cfe/runs --resume

⚠️ No hay checkpoint para reanudar en /content/drive/MyDrive/falcon_cfe/runs/cfe_satelital/weights/last.pt. Empezando de cero.
🚀 Entrenando yolov8m.pt sobre dataset.yaml
   epochs=100, imgsz=960, batch=-1, device=0, project=/content/drive/MyDrive/falcon_cfe/runs
New https://pypi.org/project/ultralytics/8.4.87 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.86 🚀 Python-3.12.13 torch-2.11.0+cpu 
Traceback (most recent call last):
  File "/content/ProyectoMEC/entrenamiento/entrenar.py", line 112, in <module>
    main()
  File "/content/ProyectoMEC/entrenamiento/entrenar.py", line 107, in main
    entrenar(args.modelo, args.epochs, args.imgsz, args.batch, args.nombre,
  File "/content/ProyectoMEC/entrenamiento/entrenar.py", line 69, in entrenar
    resultados = model.train(**kwargs)
                 ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ultralytics/engine/model.py", line 805, in train
    self.trainer = (trainer or self._smart_load("trai

## (Opcional) Ver métricas y ejemplos de detección

In [11]:
import glob
from IPython.display import Image, display
for img in glob.glob('/content/drive/MyDrive/falcon_cfe/runs/**/results.png', recursive=True):
    display(Image(filename=img))
for img in glob.glob('/content/drive/MyDrive/falcon_cfe/runs/**/val_batch0_pred.jpg', recursive=True)[:2]:
    display(Image(filename=img))